In [ ]:
#ПОЛНЫЙПОЛНЫЙ=======================================
# ПОЛНЫПОЛНЫЙЙ КОД ДЛЯ COLAB T4
# Подключение к Kaggle + обучение сегментации
# ============================================

# ============================================
# 1. УСТАНОВКА БИБЛИОТЕК
# ============================================
!pip install -q segmentation-models-pytorch timm albumentations kagglehub

# ============================================
# 2. ПОДКЛЮЧЕНИЕ К KAGGLE
# ============================================

import os
import json
from pathlib import Path

# Установка переменной окружения
os.environ["KAGGLE_API_TOKEN"] = "KGAT_8f137c9242dca95e4b63ab243fda8ba3"

# Проверка подключения
print("\n🔍 Проверка подключения к Kaggle...")
!kaggle competitions list 2>/dev/null && echo "✅ Подключение успешно!" || echo "❌ Ошибка подключения"

COMPETITION_NAME = "dl-lab-3-product-segmentation"  # Название вашего соревнования

!kaggle competitions download -c {COMPETITION_NAME}

!unzip dl-lab-3-product-segmentation.zip

# ============================================
# 4. ИМПОРТЫ
# ============================================

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
import segmentation_models_pytorch as smp
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import autocast, GradScaler
import gc


COMPETITION_NAME = "dl-lab-3-product-segmentation"

BASE_DIR = Path("data")
TRAIN_DIR = BASE_DIR / "train/images"
TRAIN_MASKS_DIR = BASE_DIR / "train/masks"
TEST_DIR = BASE_DIR / "test_images"


In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

TRAIN_DIR = Path("/content/train/images")
MASKS_DIR = Path("/content/train/masks")
OUT_DIR = Path("./seg_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def get_image_path(images_dir: Path, stem: str):
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".JPG", ".JPEG", ".PNG"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def analyze_train_distribution(images_dir=TRAIN_DIR, masks_dir=MASKS_DIR):
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)

    mask_files = sorted(masks_dir.glob("*.png"))
    rows = []

    for mask_path in tqdm(mask_files, desc="Analyzing masks"):
        img_path = get_image_path(images_dir, mask_path.stem)
        if img_path is None:
            continue

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue

        mask_bin = (mask > 127).astype(np.uint8)
        h, w = mask_bin.shape
        fg = int(mask_bin.sum())
        total = int(mask_bin.size)
        fg_ratio = fg / total if total else 0.0

        rows.append({
            "stem": mask_path.stem,
            "image_path": str(img_path),
            "mask_path": str(mask_path),
            "height": h,
            "width": w,
            "pixels_total": total,
            "pixels_fg": fg,
            "pixels_bg": total - fg,
            "fg_ratio": fg_ratio,
            "has_fg": int(fg > 0),
            "fg_pct": fg_ratio * 100.0,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("Не найдено ни одной пары image/mask.")

    df["fg_bin_10"] = pd.cut(
        df["fg_ratio"],
        bins=[-1e-9, 0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 1.0],
        labels=[
            "0",
            "(0,0.1%]",
            "(0.1,0.5%]",
            "(0.5,1%]",
            "(1,2%]",
            "(2,5%]",
            "(5,10%]",
            "(10,20%]",
            "(20%,100%]",
        ],
    )

    summary = {
        "samples": len(df),
        "images": df["image_path"].nunique(),
        "mean_fg_ratio": df["fg_ratio"].mean(),
        "median_fg_ratio": df["fg_ratio"].median(),
        "min_fg_ratio": df["fg_ratio"].min(),
        "max_fg_ratio": df["fg_ratio"].max(),
        "empty_masks": int((df["has_fg"] == 0).sum()),
        "non_empty_masks": int((df["has_fg"] == 1).sum()),
        "empty_pct": float((df["has_fg"] == 0).mean() * 100.0),
    }

    dist_by_bin = df["fg_bin_10"].value_counts(dropna=False).sort_index()
    dist_by_has_fg = df["has_fg"].value_counts().sort_index()

    df.to_csv(OUT_DIR / "train_mask_analysis_rows.csv", index=False)

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(OUT_DIR / "train_mask_analysis_summary.csv", index=False)

    dist_by_bin_df = dist_by_bin.reset_index()
    dist_by_bin_df.columns = ["fg_bin", "count"]
    dist_by_bin_df.to_csv(OUT_DIR / "train_mask_fg_bins.csv", index=False)

    dist_by_has_fg_df = dist_by_has_fg.reset_index()
    dist_by_has_fg_df.columns = ["has_fg", "count"]
    dist_by_has_fg_df.to_csv(OUT_DIR / "train_mask_has_fg.csv", index=False)

    print(summary_df.to_string(index=False))
    print("\nDistribution by mask area bins:")
    print(dist_by_bin_df.to_string(index=False))
    print("\nHas foreground distribution:")
    print(dist_by_has_fg_df.to_string(index=False))

    return df, summary

df_analysis, analysis_summary = analyze_train_distribution()

In [ ]:
import os
import json
import time
import random
from pathlib import Path
from copy import deepcopy

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import torchvision.transforms.functional as TF


os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_image_path(images_dir: Path, stem: str):
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".JPG", ".JPEG", ".PNG"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def read_image_cv2(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Cannot read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_mask_cv2(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Cannot read mask: {path}")
    return (mask > 127).astype(np.float32)


def make_area_label(fg_ratio: float):
    if fg_ratio == 0:
        return 0
    elif fg_ratio < 0.001:
        return 1
    elif fg_ratio < 0.005:
        return 2
    elif fg_ratio < 0.01:
        return 3
    elif fg_ratio < 0.02:
        return 4
    elif fg_ratio < 0.05:
        return 5
    elif fg_ratio < 0.10:
        return 6
    elif fg_ratio < 0.20:
        return 7
    else:
        return 8


def build_samples(images_dir, masks_dir):
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    samples = []
    for mask_path in sorted(masks_dir.glob("*.png")):
        img_path = get_image_path(images_dir, mask_path.stem)
        if img_path is not None:
            samples.append((img_path, mask_path))
    if not samples:
        raise RuntimeError("No paired image/mask samples found")
    return samples


def build_stratified_folds(samples, n_splits=5, seed=42, out_dir="./seg_analysis"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    labels = []
    rows = []

    for img_path, mask_path in tqdm(samples, desc="Preparing stratification"):
        mask = read_mask_cv2(mask_path)
        fg_ratio = float(mask.mean())
        label = make_area_label(fg_ratio)
        labels.append(label)
        rows.append({
            "image_path": str(img_path),
            "mask_path": str(mask_path),
            "fg_ratio": fg_ratio,
            "label": label,
        })

    labels = np.array(labels)
    idx = np.arange(len(samples))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = []
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(idx, labels)):
        folds.append((tr_idx.tolist(), va_idx.tolist()))
        tr_labels = labels[tr_idx]
        va_labels = labels[va_idx]
        fold_rows.append({
            "fold": fold,
            "train_size": len(tr_idx),
            "val_size": len(va_idx),
            "train_hist": json.dumps({int(k): int(v) for k, v in zip(*np.unique(tr_labels, return_counts=True))}, ensure_ascii=False),
            "val_hist": json.dumps({int(k): int(v) for k, v in zip(*np.unique(va_labels, return_counts=True))}, ensure_ascii=False),
        })

    pd.DataFrame(rows).to_csv(out_dir / "sample_meta.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(out_dir / "folds.csv", index=False)
    print(pd.DataFrame(fold_rows).to_string(index=False))
    return folds


class SegTransform:
    def __init__(self, img_size=380, train=True):
        self.img_size = img_size
        self.train = train
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __call__(self, image, mask):
        image = torch.from_numpy(image).float() / 255.0
        mask = torch.from_numpy(mask).float()

        image = image.permute(2, 0, 1)
        mask = mask.unsqueeze(0)

        image = TF.resize(image, [self.img_size, self.img_size], antialias=True)
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = (mask > 0.5).float()

        if self.train:
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            if random.random() < 0.2:
                image = TF.vflip(image)
                mask = TF.vflip(mask)
            if random.random() < 0.35:
                angle = random.uniform(-20, 20)
                image = TF.rotate(image, angle, fill=0)
                mask = TF.rotate(mask, angle, fill=0)
            if random.random() < 0.35:
                image = TF.adjust_brightness(image, random.uniform(0.85, 1.15))
                image = TF.adjust_contrast(image, random.uniform(0.85, 1.15))
                image = TF.adjust_saturation(image, random.uniform(0.85, 1.15))
            if random.random() < 0.15:
                image = torch.clamp(image + torch.randn_like(image) * 0.03, 0, 1)

        image = (image - self.mean) / self.std
        return image, mask


class BinarySegDataset(Dataset):
    def __init__(self, samples, img_size=380, train=True):
        self.samples = samples
        self.tf = SegTransform(img_size=img_size, train=train)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]
        image = read_image_cv2(image_path)
        mask = read_mask_cv2(mask_path)
        image, mask = self.tf(image, mask)
        return image, mask


class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.flatten(1)
        targets = targets.flatten(1)
        inter = (probs * targets).sum(dim=1)
        denom = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2 * inter + self.eps) / (denom + self.eps)
        return 1 - dice.mean()


class ComboLoss(nn.Module):
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_w = bce_w
        self.dice_w = dice_w

    def forward(self, logits, targets):
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(logits, size=targets.shape[-2:], mode="bilinear", align_corners=False)
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)


def dice_score_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    if preds.shape[-2:] != targets.shape[-2:]:
        preds = F.interpolate(preds, size=targets.shape[-2:], mode="nearest")
    preds = preds.flatten(1)
    targets = targets.flatten(1)
    inter = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * inter + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    if preds.shape[-2:] != targets.shape[-2:]:
        preds = F.interpolate(preds, size=targets.shape[-2:], mode="nearest")
    preds = preds.flatten(1)
    targets = targets.flatten(1)
    inter = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - inter
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class EfficientNetB4UNet(nn.Module):
    def __init__(self, encoder_name="tf_efficientnet_b4.ns_jft_in1k", pretrained=True, img_size=380):
        super().__init__()
        self.encoder = timm.create_model(
            encoder_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            feats = self.encoder(dummy)
            chs = [f.shape[1] for f in feats]

        self.center = ConvBlock(chs[4], 256)
        self.dec4 = DecoderBlock(256, chs[3], 192)
        self.dec3 = DecoderBlock(192, chs[2], 128)
        self.dec2 = DecoderBlock(128, chs[1], 96)
        self.dec1 = DecoderBlock(96, chs[0], 64)
        self.head = nn.Sequential(
            ConvBlock(64, 32),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        inp_size = x.shape[-2:]
        feats = self.encoder(x)
        x = self.center(feats[4])
        x = self.dec4(x, feats[3])
        x = self.dec3(x, feats[2])
        x = self.dec2(x, feats[1])
        x = self.dec1(x, feats[0])
        x = self.head(x)
        if x.shape[-2:] != inp_size:
            x = F.interpolate(x, size=inp_size, mode="bilinear", align_corners=False)
        return x


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for k, v in model.state_dict().items():
            self.shadow[k] = v.detach().clone()

    @torch.no_grad()
    def update(self, model):
        sd = model.state_dict()
        for k, v in sd.items():
            if k not in self.shadow:
                self.shadow[k] = v.detach().clone()
                continue
            if torch.is_floating_point(v):
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def apply_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


@torch.no_grad()
def predict_tta(model, images):
    logits = model(images)
    logits_flip = model(torch.flip(images, dims=[3]))
    logits_flip = torch.flip(logits_flip, dims=[3])
    return 0.5 * (logits + logits_flip)


def train_one_epoch(model, loader, optimizer, loss_fn, scaler, ema, threshold=0.5):
    model.train()
    total_loss = total_dice = total_iou = 0.0
    pbar = tqdm(loader, desc="train", leave=False)

    for images, masks in pbar:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss = loss_fn(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = loss_fn(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        if ema is not None:
            ema.update(model)

        total_loss += loss.item()
        total_dice += dice_score_from_logits(logits.detach(), masks, threshold=threshold)
        total_iou += iou_score_from_logits(logits.detach(), masks, threshold=threshold)
        pbar.set_postfix(loss=f"{total_loss/(len(pbar)+1e-9):.4f}")

    n = len(loader)
    return {"loss": total_loss / n, "dice": total_dice / n, "iou": total_iou / n}


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, threshold=0.5, use_tta=True):
    model.eval()
    total_loss = total_dice = total_iou = 0.0
    for images, masks in tqdm(loader, desc="valid", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        logits = predict_tta(model, images) if use_tta else model(images)
        loss = loss_fn(logits, masks)
        total_loss += loss.item()
        total_dice += dice_score_from_logits(logits, masks, threshold=threshold)
        total_iou += iou_score_from_logits(logits, masks, threshold=threshold)
    n = len(loader)
    return {"loss": total_loss / n, "dice": total_dice / n, "iou": total_iou / n}


@torch.no_grad()
def infer_test_ensemble(model_paths, test_dir, img_size=380, threshold=0.5):
    test_dir = Path(test_dir)
    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        imgs.extend(test_dir.glob(ext))
    imgs = sorted(imgs)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    models = []
    for p in model_paths:
        model = EfficientNetB4UNet(pretrained=False, img_size=img_size).to(DEVICE)
        ckpt = torch.load(p, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        models.append(model)

    results = []
    for p in tqdm(imgs, desc="test", leave=False):
        img = read_image_cv2(p)
        oh, ow = img.shape[:2]
        x = torch.from_numpy(img).float() / 255.0
        x = x.permute(2, 0, 1)
        x = TF.resize(x, [img_size, img_size], antialias=True)
        x = (x - mean) / std
        x = x.unsqueeze(0).to(DEVICE)

        prob = 0.0
        for model in models:
            logits = predict_tta(model, x)
            prob += torch.sigmoid(logits)[0, 0].cpu().numpy()
        prob /= len(models)

        prob = cv2.resize(prob, (ow, oh), interpolation=cv2.INTER_LINEAR)
        mask = (prob > threshold).astype(np.uint8)
        results.append({"ImageId": p.name, "mask": json.dumps(mask.tolist(), separators=(",", ":"))})
    return pd.DataFrame(results)


def train_fold(fold, train_samples, val_samples, save_dir, cfg, img_size):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    train_ds = BinarySegDataset(train_samples, img_size=img_size, train=True)
    val_ds = BinarySegDataset(val_samples, img_size=img_size, train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["BATCH_SIZE"],
        shuffle=True,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=(DEVICE == "cuda"),
        drop_last=True,
        persistent_workers=(cfg["NUM_WORKERS"] > 0),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["BATCH_SIZE"],
        shuffle=False,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
        persistent_workers=(cfg["NUM_WORKERS"] > 0),
    )

    model = EfficientNetB4UNet(pretrained=True, img_size=img_size).to(DEVICE)
    loss_fn = ComboLoss(bce_w=0.5, dice_w=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WD"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    ema = EMA(model, decay=0.999)

    best_dice = -1.0
    bad_epochs = 0
    history = []

    for epoch in range(1, cfg["EPOCHS"] + 1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler if DEVICE == "cuda" else None, ema, threshold=cfg["THRESHOLD"])

        shadow_model = deepcopy(model)
        ema.apply_to(shadow_model)
        va = validate_one_epoch(shadow_model, val_loader, loss_fn, threshold=cfg["THRESHOLD"], use_tta=True)

        scheduler.step()

        row = {
            "fold": fold,
            "img_size": img_size,
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": tr["loss"],
            "train_dice": tr["dice"],
            "train_iou": tr["iou"],
            "val_loss": va["loss"],
            "val_dice": va["dice"],
            "val_iou": va["iou"],
            "time_sec": time.time() - t0,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(save_dir / "history.csv", index=False)

        ckpt = {
            "epoch": epoch,
            "model": model.state_dict(),
            "ema": ema.shadow,
            "optimizer": optimizer.state_dict(),
            "history": history,
            "config": cfg,
            "img_size": img_size,
            "fold": fold,
        }
        torch.save(ckpt, save_dir / "last.pth")

        if va["dice"] > best_dice:
            best_dice = va["dice"]
            bad_epochs = 0
            torch.save(ckpt, save_dir / "best.pth")
        else:
            bad_epochs += 1

        print(
            f"Size {img_size} | Fold {fold} | Epoch {epoch:03d} | "
            f"train loss {tr['loss']:.4f} dice {tr['dice']:.4f} iou {tr['iou']:.4f} | "
            f"val loss {va['loss']:.4f} dice {va['dice']:.4f} iou {va['iou']:.4f} | "
            f"best {best_dice:.4f}"
        )

        if bad_epochs >= cfg["PATIENCE"]:
            print(f"Size {img_size} | Fold {fold}: early stopping.")
            break

    return best_dice


def main():
    seed_everything(42)

    TRAIN_DIR = Path("/content/train/images")
    MASKS_DIR = Path("/content/train/masks")
    TEST_DIR = Path("/content/test_images")
    OUT_DIR = Path("./seg_effb4_multiscale")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    cfg = {
        "IMG_SIZES": [256, 320, 384, 512, 640],  # Разные размеры изображений
        "BATCH_SIZE": 16,
        "EPOCHS": 40,
        "LR": 3e-4,
        "WD": 1e-4,
        "N_FOLDS": 5,
        "NUM_WORKERS": 4,
        "THRESHOLD": 0.5,
        "PATIENCE": 8,
    }

    samples = build_samples(TRAIN_DIR, MASKS_DIR)
    folds = build_stratified_folds(samples, n_splits=cfg["N_FOLDS"], seed=42, out_dir="./seg_analysis")

    all_results = []
    all_model_paths = []
    models_by_size = {size: [] for size in cfg["IMG_SIZES"]}

    # Цикл по разным размерам изображений
    for img_size in cfg["IMG_SIZES"]:
        print(f"\n{'='*60}")
        print(f"Training with image size: {img_size}x{img_size}")
        print(f"{'='*60}\n")

        size_dir = OUT_DIR / f"size_{img_size}"
        size_dir.mkdir(parents=True, exist_ok=True)

        fold_scores = []

        # Цикл по фолдам для текущего размера
        for fold in range(cfg["N_FOLDS"]):
            train_idx, val_idx = folds[fold]
            train_samples = [samples[i] for i in train_idx]
            val_samples = [samples[i] for i in val_idx]

            fold_dir = size_dir / f"fold_{fold}"
            cfg_fold = dict(cfg)
            cfg_fold["fold"] = fold
            cfg_fold["IMG_SIZE"] = img_size

            score = train_fold(fold, train_samples, val_samples, fold_dir, cfg_fold, img_size)

            fold_scores.append({
                "img_size": img_size,
                "fold": fold,
                "best_dice": score
            })

            model_path = fold_dir / "best.pth"
            models_by_size[img_size].append(model_path)
            all_model_paths.append(model_path)

            all_results.append({
                "img_size": img_size,
                "fold": fold,
                "best_dice": score
            })

        # Сохраняем результаты для текущего размера
        pd.DataFrame(fold_scores).to_csv(size_dir / "fold_scores.csv", index=False)

        # Средний score по фолдам для текущего размера
        mean_dice = np.mean([s["best_dice"] for s in fold_scores])
        print(f"\nSize {img_size}: Mean CV Dice = {mean_dice:.4f}\n")

    # Сохраняем все результаты
    pd.DataFrame(all_results).to_csv(OUT_DIR / "all_results.csv", index=False)

    # Выводим сводную таблицу
    summary = pd.DataFrame(all_results).groupby('img_size')['best_dice'].agg(['mean', 'std', 'max']).round(4)
    print("\n" + "="*60)
    print("Summary by image size:")
    print("="*60)
    print(summary.to_string())
    summary.to_csv(OUT_DIR / "summary_by_size.csv")

    # Сохраняем пути ко всем моделям
    with open(OUT_DIR / "all_model_paths.txt", "w") as f:
        for p in all_model_paths:
            f.write(f"{p}\n")

    # Инференс на тесте с использованием ансамбля всех моделей на их родных размерах
    if TEST_DIR.exists():
        print("\n" + "="*60)
        print("Running inference on test set with multi-scale ensemble...")
        print("="*60)

        sub = infer_test_multiscale_ensemble(
            models_by_size,
            TEST_DIR,
            threshold=cfg["THRESHOLD"]
        )
        sub.to_csv(OUT_DIR / "submission_multiscale.csv", index=False)
        print(f"Saved submission to {OUT_DIR / 'submission_multiscale.csv'}")


@torch.no_grad()
def infer_test_multiscale_ensemble(models_by_size, test_dir, threshold=0.5):
    """
    Инференс с использованием ансамбля моделей, обученных на разных размерах.
    Каждая модель предсказывает на том размере, на котором училась.
    """
    test_dir = Path(test_dir)
    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        imgs.extend(test_dir.glob(ext))
    imgs = sorted(imgs)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    # Загружаем все модели, группируя их по размерам
    all_models = []
    for img_size, paths in models_by_size.items():
        for model_path in paths:
            model = EfficientNetB4UNet(pretrained=False, img_size=img_size).to(DEVICE)
            ckpt = torch.load(model_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model"])
            model.eval()
            all_models.append({
                'model': model,
                'img_size': img_size
            })

    print(f"Loaded {len(all_models)} models: {len(models_by_size)} sizes × {len(next(iter(models_by_size.values())))} folds")

    results = []
    for img_path in tqdm(imgs, desc="Multi-scale inference"):
        img = read_image_cv2(img_path)
        oh, ow = img.shape[:2]

        # Аккумулируем вероятности со всех моделей
        accumulated_probs = []

        for model_info in all_models:
            model = model_info['model']
            img_size = model_info['img_size']

            # Подготовка изображения для конкретного размера модели
            x = torch.from_numpy(img).float() / 255.0
            x = x.permute(2, 0, 1)
            x = TF.resize(x, [img_size, img_size], antialias=True)
            x = (x - mean) / std
            x = x.unsqueeze(0).to(DEVICE)

            # Предсказание с TTA
            logits = predict_tta(model, x)
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

            # Ресайз предсказания обратно к оригинальному размеру
            prob_resized = cv2.resize(prob, (ow, oh), interpolation=cv2.INTER_LINEAR)
            accumulated_probs.append(prob_resized)

        # Усредняем вероятности со всех моделей
        final_prob = np.mean(accumulated_probs, axis=0)
        mask = (final_prob > threshold).astype(np.uint8)

        results.append({
            "ImageId": img_path.name,
            "mask": json.dumps(mask.tolist(), separators=(",", ":"))
        })

    return pd.DataFrame(results)


if __name__ == "__main__":
    main()


In [ ]:
import os

# Путь к исходной папке
SOURCE_DIR = "/content"

# Файлы и папки для исключения (те же, что и в коде переноса)
EXCLUDE = [
    "dl-lab-3-product-segmentation.zip",
    "train",
    "test_images",
    "seg_analysis",
    "sample_data",
    ".config",
    "drive",
    "unlabeled"
]

def should_exclude(name):
    """Проверяем, нужно ли исключить файл/папку"""
    for exclude in EXCLUDE:
        if name == exclude or name.startswith(exclude):
            return True
    return False

def get_folder_size(path):
    """Рекурсивно подсчитывает размер папки в байтах"""
    total = 0
    try:
        for entry in os.scandir(path):
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_folder_size(entry.path)
    except (PermissionError, OSError):
        pass
    return total

def format_size(size_bytes):
    """Форматирует размер в читаемый вид"""
    if size_bytes == 0:
        return "0 B"
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} TB"

print("📊 Анализ размера данных ДО переноса")
print("=" * 60)
print(f"📂 Исходная папка: {SOURCE_DIR}")
print(f"🚫 Исключения: {EXCLUDE}")
print("-" * 60)

total_size = 0
items_to_copy = []
items_to_exclude = []

# Анализируем каждый элемент
for item in os.listdir(SOURCE_DIR):
    item_path = os.path.join(SOURCE_DIR, item)

    if should_exclude(item):
        if os.path.isdir(item_path):
            size = get_folder_size(item_path)
        else:
            size = os.path.getsize(item_path)
        items_to_exclude.append((item, size))
    else:
        if os.path.isdir(item_path):
            size = get_folder_size(item_path)
        else:
            size = os.path.getsize(item_path)
        items_to_copy.append((item, size))
        total_size += size

# Выводим что БУДЕТ скопировано
print("\n✅ БУДЕТ скопировано:")
print("-" * 40)
for item, size in sorted(items_to_copy, key=lambda x: x[1], reverse=True):
    if os.path.isdir(os.path.join(SOURCE_DIR, item)):
        print(f"  📁 {item}: {format_size(size)}")
    else:
        print(f"  📄 {item}: {format_size(size)}")

# Выводим что НЕ БУДЕТ скопировано
print("\n⏭️ НЕ БУДЕТ скопировано (исключения):")
print("-" * 40)
for item, size in sorted(items_to_exclude, key=lambda x: x[1], reverse=True):
    if os.path.isdir(os.path.join(SOURCE_DIR, item)):
        print(f"  📁 {item}: {format_size(size)}")
    else:
        print(f"  📄 {item}: {format_size(size)}")

# Итоги
print("\n" + "=" * 60)
print("📊 ИТОГИ:")
print("-" * 60)
print(f"📦 Общий размер данных ДЛЯ КОПИРОВАНИЯ: {format_size(total_size)}")
print(f"📁 Количество элементов для копирования: {len(items_to_copy)}")
print(f"⏭️ Количество исключенных элементов: {len(items_to_exclude)}")

# Конвертация в MB и GB для удобства
print(f"\n💾 Подробно:")
print(f"   {total_size / (1024*1024):.2f} MB")
print(f"   {total_size / (1024*1024*1024):.2f} GB")

# Предупреждение, если размер большой
if total_size > 500 * 1024 * 1024:  # больше 500 MB
    print(f"\n⚠️ ВНИМАНИЕ: Размер данных превышает 500 MB!")
    print("   Копирование может занять несколько минут.")

if total_size > 2 * 1024 * 1024 * 1024:  # больше 2 GB
    print(f"\n⚠️⚠️ СЕРЬЕЗНОЕ ВНИМАНИЕ: Размер данных превышает 2 GB!")
    print("   Убедитесь, что на Google Диске достаточно места.")

In [ ]:
from google.colab import drive
import shutil
import os
import fnmatch

# Монтируем Google Диск
drive.mount('/content/drive')

# Пути
SOURCE_DIR = "/content"  # исходная папка в Colab
DEST_DIR = "/content/drive/MyDrive/size"  # папка назначения на Диске

# Создаём папку назначения
os.makedirs(DEST_DIR, exist_ok=True)

# Файлы и папки для исключения
EXCLUDE = [
    "dl-lab-3-product-segmentation.zip",
    "train",  # исключаем папку train
    "test_images",  # исключаем странное название (возможно опечатка)
    "sample_data",  # стандартная папка Colab
    ".config",
    "seg_analysis",
    "drive",
    "unlabeled" # не копируем саму папку drive
]

# Паттерны для исключения файлов (поддержка wildcard)
EXCLUDE_PATTERNS = [
    "*last.pth",      # исключаем все last.pth файлы
    "*checkpoint*",   # исключаем любые checkpoint файлы (опционально)
    "*.tmp",          # временные файлы
    "*.pyc",          # скомпилированные Python файлы
]

def should_exclude(name, full_path=None):
    """Проверяем, нужно ли исключить файл/папку"""
    # Проверяем точные совпадения
    for exclude in EXCLUDE:
        if name == exclude or name.startswith(exclude):
            return True

    # Проверяем паттерны для файлов
    if full_path and os.path.isfile(full_path):
        for pattern in EXCLUDE_PATTERNS:
            if fnmatch.fnmatch(name, pattern):
                return True

    return False

def get_size(path):
    """Подсчитывает размер папки или файла"""
    if os.path.isfile(path):
        return os.path.getsize(path)
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            if os.path.exists(filepath):
                total += os.path.getsize(filepath)
    return total

def copy_with_exclusion(src, dst, exclude_patterns=None):
    """
    Рекурсивное копирование с исключением файлов по паттернам
    """
    if exclude_patterns is None:
        exclude_patterns = EXCLUDE_PATTERNS

    # Создаем папку назначения
    os.makedirs(dst, exist_ok=True)

    copied_count = 0
    total_size = 0

    for item in os.listdir(src):
        src_path = os.path.join(src, item)
        dst_path = os.path.join(dst, item)

        # Проверяем, нужно ли исключить
        if should_exclude(item, src_path):
            print(f"⏭️ Пропущено: {src_path}")
            continue

        try:
            if os.path.isdir(src_path):
                # Рекурсивно копируем папку
                sub_copied, sub_size = copy_with_exclusion(src_path, dst_path, exclude_patterns)
                copied_count += sub_copied
                total_size += sub_size
                print(f"✅ Папка: {item} ({sub_size / 1024 / 1024:.2f} MB)")
            else:
                # Копируем файл
                shutil.copy2(src_path, dst_path)
                file_size = os.path.getsize(src_path)
                total_size += file_size
                copied_count += 1
                print(f"✅ Файл: {item} ({file_size / 1024 / 1024:.2f} MB)")

        except Exception as e:
            print(f"❌ Ошибка при копировании {src_path}: {e}")

    return copied_count, total_size

print("📦 Начинаю копирование...")
print(f"Источник: {SOURCE_DIR}")
print(f"Назначение: {DEST_DIR}")
print(f"Исключенные папки/файлы: {EXCLUDE}")
print(f"Исключенные паттерны: {EXCLUDE_PATTERNS}")
print("-" * 50)

# Запускаем копирование
copied_count, total_size = copy_with_exclusion(SOURCE_DIR, DEST_DIR)

print("-" * 50)
print(f"\n✅ Копирование завершено!")
print(f"📁 Скопировано элементов: {copied_count}")
print(f"💾 Общий размер: {total_size / 1024 / 1024:.2f} MB")
print(f"📍 Расположение: {DEST_DIR}")

# Показываем содержимое скопированной папки
print(f"\n📋 Содержимое {DEST_DIR}:")
def list_contents(path, indent=0):
    for item in sorted(os.listdir(path)):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            print(f"{'  ' * indent}📁 {item}/")
            list_contents(item_path, indent + 1)
        else:
            size = os.path.getsize(item_path) / 1024 / 1024
            # Пропускаем отображение last.pth в выводе (на всякий случай)
            if not item.endswith('last.pth'):
                print(f"{'  ' * indent}📄 {item} ({size:.2f} MB)")

list_contents(DEST_DIR)

# Дополнительно: поиск и удаление last.pth в уже скопированных данных (на всякий случай)
print("\n🔍 Проверка на наличие last.pth в скопированных данных...")
found_last_pth = []
for root, dirs, files in os.walk(DEST_DIR):
    for file in files:
        if file == "last.pth" or file.endswith("last.pth"):
            found_last_pth.append(os.path.join(root, file))

if found_last_pth:
    print(f"⚠️ Найдено {len(found_last_pth)} файлов last.pth:")
    for p in found_last_pth:
        print(f"  - {p}")
        os.remove(p)
        print(f"    🗑️ Удалён")
    print("✅ Все last.pth файлы удалены")
else:
    print("✅ Файлов last.pth не найдено")